In [ ]:
# imports

import math

import numpy as np
import pandas as pd
from bokeh.io import output_notebook

from omni_drive import (
    Angle,
    Clock,
    Distance,
    DriveVector,
    Kalman2d,
    OmniDrive,
    PoseRecorder,
    SensorRecorder,
    Time,
)
from omni_drive.live import LivePlot

output_notebook(hide_banner=True)


In [ ]:
robot = OmniDrive()

In [ ]:
robot.print_status()

## Część 1: Ruch i odometria

**Zadanie 1 – test dryfu odometrii**

<small>Sprawdźmy, jak odometria zachowuje się w praktyce. Wykonaj test dryfu kół, realizując przejazd po zamkniętej trasie kwadratu. Uzupełnij funkcję `drive_square`:</small>


In [ ]:
def drive_square():
    robot.drive_for(...)
    robot.pause_for(...)
    ...

robot.reset_pose()

for i in range(1):  # zwiększ liczbę przejść gdy uzupełnisz definicję drive_square
    drive_square()
    print(f"Przejście nr {i}: Odometria: {robot.pose}")


<div class="alert alert-block alert-info">
  <small><b>Pytanie:</b> Czy robot wrócił w miejsce początkowe? Jak daleko znalazł się od niego?</small>
</div>

### Położenie na podstawie LiDARu

<small>Jak już wiesz, oprócz trzech par czujników odległości zamontowanych co 90°, robot posiada również skaner laserowy LiDAR, który mierzy odległość do 680 punktów w polu widzenia 240°.</small>

<small>Będziemy korzystać z pomiarów LiDAR-u dokładnie w tych samych kierunkach, w które skierowane są pozostałe sensory. Biblioteka udostępnia w tym celu gotowe właściwości API, które wyznaczają średnią arytmetyczną z 5 punktów lasera w osi każdego z czujników:</small>

* <small>`robot.lidar.beam_left` – wiązka w lewo (+90°),</small>
* <small>`robot.lidar.beam_front` – wiązka na wprost (0°),</small>
* <small>`robot.lidar.beam_right` – wiązka w prawo (-90°).</small>

<small>Uśrednienie kilku sąsiednich punktów skutecznie redukuje szum pojedynczego pomiaru lasera.</small>

<small>Znając przesunięcie robota względem trzech punktów pomiarowych (ścian: lewej, przedniej i prawej) potrafimy z dużą dokładnością określić jego pozycję w osi Y oraz trochę mniej dokładnie w osi X:</small>

<div style="display: flex; gap: 20px; justify-content: center; align-items: flex-start; margin: 20px 0;">
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_start.svg" alt="Pozycja startowa" style="width: 100%; height: auto;" />
  </div>
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_moved.svg" alt="Po przesunięciu" style="width: 100%; height: auto;" />
  </div>
</div>

_Gdy robot przesunie się o $x$ do przodu, odległość do przedniej ściany zmaleje o $x$, a gdy przesunie się o $y$ w lewo, odległość do lewej ściany zmaleje o $y$, a do prawej wzrośnie o $y$. Położenie względem startu to zatem:_

$$
x = f_0 - f, \qquad y = \frac{(l_0 - l) - (r_0 - r)}{2},
$$

<small>gdzie:</small>

* $f_0$, $l_0$, $r_0$ <small>to odległości do przedniej, lewej i prawej ściany zmierzone na starcie (przed ruszeniem robota),</small>
* $f$, $l$, $r$ <small>to aktualne odległości odczytywane w trakcie jazdy z wiązek LiDAR-u (odpowiednio: `beam_front`, `beam_left`, `beam_right`).</small>


<div class="alert alert-block alert-warning">
  <small>
  <b>Uwaga:</b> Takie położenie nie dryfuje, ponieważ ściany się nie przesuwają, ale uproszczenie to działa tylko w znanym, prostokątnym otoczeniu i przy stałej orientacji robota. W praktyce roboty zwykle poruszają się w dowolnym otoczeniu – budują wtedy z kolejnych skanów mapę i dopasowują do niej każdy nowy skan (scan matching); zasada pozostaje ta sama, ale ściany i punkty odniesienia są wyznaczane bezpośrednio z danych.
  </small>
</div>


**Zadanie 2 - odometria czy LiDAR**

<small>Wykorzystamy funkcję `drive_square` z poprzedniego ćwiczenia. Wstaw robota do prostokątnego kartonu i dostosuj długość przejazdu, tak, żeby poruszał się w środku nie obijając się o ściany.</small>

<small>Wykonamy trzy przejazdy, będziemy analizować pozycję robota raportowaną przez odometrię oraz obliczaną na podstawie danych z LiDAR-u.</small>

<small>Zaimplementuj logikę pozwalającą wyznaczać pozycję x oraz y za pomocą LiDARu.</small>


In [ ]:
f0, l0, r0 = None, None, None

def lidar_reset():
    global f0, l0, r0
    f0 = robot.lidar.beam_front
    l0 = robot.lidar.beam_left
    r0 = robot.lidar.beam_right

def lidar_get_x():
    f = robot.lidar.beam_front
    dx = ...
    return Distance.m(dx)

def lidar_get_y():
    ...
    dy = ...
    return Distance.m(dy)


In [ ]:
robot.reset_pose()
lidar_reset()

rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria vs LiDAR"
)

def print_pos(label: str):
    pose = robot.pose
    px, py = pose.x.meters, pose.y.meters
    lx, ly = lidar_get_x().meters, lidar_get_y().meters

    d = math.hypot(px - lx, py - ly)
    print(f"{label}: Odometria: ({px:.3f}, {py:.3f}), LiDAR: ({lx:.3f}, {ly:.3f}), Różnica: {d:.3f} m")
    return d

print_pos("Pozycja startowa")

final_drift = 0.0
with LivePlot(rec), robot:
    for i in range(3):
        drive_square()

        # Sprawdzamy wartości oraz różnicę po każdym przejściu pętli
        final_drift = print_pos(f"Przejście nr {i + 1}")


<div class="alert alert-block alert-info">
  <small><b>Pytania:</b> Co jest dokładniejsze: odometria czy pozycja obliczona na podstawie LiDAR-u? Jak duża jest różnica? Dlaczego na zamkniętej trasie część błędu odometrii się znosi?</small>
</div>

<div class="alert alert-block alert-success">
  <small><b>Wskazówka:</b> Pozycja z LiDAR-u to nasz niezależny punkt odniesienia. Nie używamy jej do obliczeń &ndash; służy wyłącznie na wykresach do oceny błędu odometrii i skuteczności kolejnych algorytmów.</small>
</div>

**Czym jest szum procesu $Q$?**

<small>Zaobserwowana w teście różnica między odometrią a LiDAR-em na mecie (`final_drift`) to wynik kumulacji drobnych poślizgów kół na każdym kroku pętli sterowania. Pojedynczy błąd na jeden krok pętli (odometrii) jest rzędu ułamka milimetra, ale w miarę upływu czasu błędy sumują się, powodując narastający dryf pozycji robota.</small>

<small>W modelu ruchu robota tę losową niepewność opisujemy jako <b>wariancję szumu procesu</b> </small>$Q$<small> przypadającą na pojedynczy krok odometrii. Skoro wariancje niezależnych błędów sumują się z każdym krokiem, po </small>$N$<small> próbkach zarejestrowanych przez `rec` całkowita wariancja dryfu wynosi:</small>

$$\sigma^2_{\text{całk}} \approx N \cdot Q \implies Q \approx \frac{\Delta_{\text{dryf}}^2}{N}$$

<small>Wykorzystaj zarejestrowaną liczbę kroków </small>$N$<small> z obiektu `rec.data` oraz końcowy dryf `final_drift`, aby obliczyć wariancję szumu procesu </small>$Q$<small> na pojedynczy krok:</small>


In [ ]:
N_steps = len(rec.data["odometry"])

Q = ...

print(f"Liczba próbek z rec:            N = {N_steps}")
print(f"Zmierzony dryf na mecie:       Δ = {final_drift * 1000:.1f} mm")
print(f"Oszacowany szum procesu:      √Q = {math.sqrt(Q) * 1000:.2f} mm / krok")
print(f"Wariancja szumu procesu:       Q = {Q:.6f} m²  ({Q * 1_000_000:.2f} mm²)")


## Część 2: Czujniki odległości i Filtr Kalmana

**Zadanie 3 – Ruch wahadłowy**

<small>Skupimy się teraz na pozostałych czujnikach, aby zobaczyć, jak radzą sobie w porównaniu do LiDARu.</small>

<small>Zaimplementuj płynny ruch wahadłowy robota pod kątem 15° w kierunku ściany i z powrotem, tak aby wykonał dwa pełne okresy oscylacji. Do sterowania prędkością wykorzystaj funkcję okresową:</small>

$$v(t) = A \cdot \sin\left(\frac{2\pi}{T} \cdot t\right), \qquad T = \frac{t_{\text{całk}}}{n}$$


In [ ]:
robot.reset_pose()

clock = Clock(duration=Time.s(10))

rec = SensorRecorder(
    clock,
    title="Ruch wahadłowy: LiDAR vs US vs IR",
    styles={"ultrasonic": "scatter", "ir": "scatter", "lidar": "dashed"},
    lidar=lambda: robot.lidar.beam_front.meters,
    ultrasonic=lambda: robot.us_front.distance.meters,
    ir=lambda: robot.ir_front.distance.meters,
)

SPEED = ...   # A
PERIOD = ...  # T

with LivePlot(rec), robot:
    for t in clock:
        ...
        robot.set_vector(DriveVector(...))


<div class="alert alert-block alert-info">
  <small><b>Prześledź wykresy:</b> Co na nich widać? Który z odczytów jest najdokładniejszy, a który najmniej dokladny?</small>
</div>

**Czym jest szum pomiaru $R$?**

<small>Każdy czujnik jest obarczony losową niedoskonałością aparatury, którą opisujemy statystycznie jako <b>wariancję szumu pomiaru</b> </small>$R$<small>. Wykorzystamy pomiary z Zadania 3 (`rec.data`), aby wyznaczyć szum pomiarowy. Jako punkt odniesienia wykorzystamy LiDAR:</small>

$$e = z_{\text{sensor}} - z_{\text{lidar}}$$

<small>Zamiast zwykłej wariancji (`np.var()`), która jest bezbronna wobec pojedynczych fałszywych ech czujnika ultradźwiękowego (outlierów), stosujemy odporną estymację <b>MAD</b> (odchylenie bezwzględne od mediany):</small>

$$\text{MAD} = \operatorname{median}(|e - \operatorname{median}(e)|), \qquad \sigma \approx 1{,}4826 \cdot \text{MAD}, \qquad R = \sigma^2$$

<small>Wyznaczone w ten sposób wartości </small>$R_{\text{ir}}$<small> oraz </small>$R_{\text{us}}$<small> posłużą jako wagi określające zaufanie do poszczególnych sensorów.</small>

In [ ]:
df = pd.DataFrame(rec.data)
df.head()


In [ ]:
def calc_R(series):
    clean = series.dropna()  # odrzucenie braków pomiaru (np. brak echa, poza zakresem)
    mad = ...  # użyj np.abs, np.median
    return max((1.4826 * mad) ** 2, 1e-6)


In [ ]:
# Wyznacz szum pomiarowy R dla czujników IR i Ultrasonic względem referencyjnego LiDAR-u:
R_ir = ...
R_us = ...

print("\n--- Odporna estymacja szumu losowego R (MAD) ---")
print(f"IR:         √R = {math.sqrt(R_ir) * 1000:5.1f} mm")
print(f"Ultrasonic: √R = {math.sqrt(R_us) * 1000:5.1f} mm")


### Filtr Kalmana

<small>Odometria jest płynna, lecz <b>dryfuje</b> (błąd akumuluje się w czasie). Czujniki odległości (ultradźwiękowy, IR) nie dryfują, ale są <b>zaszumione</b>.</small>

<small><b>Filtr Kalmana</b> łączy oba źródła, wyznaczając najbardziej prawdopodobny <b>stan robota</b> </small>$x$<small>, wykonując naprzemiennie dwa kroki:</small>

---

#### 1. Krok predykcji (ruch robota z odometrii)
<small>Przesuwamy stan robota o przemieszczenie z kół </small>$\Delta x$<small>. Niepewność stanu (wariancja </small>$P$<small>) rośnie o szum procesu </small>$Q$<small>:</small>
$$x_{\text{pred}} = x + \Delta x, \qquad P_{\text{pred}} = P + Q$$

#### 2. Krok korekty (pomiar pozycji $z$)
<small>Pomiar </small>$z$<small> musi opisywać tę samą wielkość co stan </small>$x$<small>. Ze zmierzonego dystansu do ściany </small>$d$<small> wyznaczamy położenie robota jako zbliżenie względem startu (</small>$d_0$<small>):</small>
$$z = d_0 - d$$

<small>Obliczamy innowację (błąd prognozy) </small>$y$<small> oraz <b>wzmocnienie Kalmana</b> </small>$K \in [0, 1]$<small>, które określa stopień zaufania do pomiaru:</small>
$$y = z - x_{\text{pred}}, \qquad K = \frac{P_{\text{pred}}}{P_{\text{pred}} + R}$$
<small>Korygujemy stan robota oraz redukujemy jego niepewność:</small>
$$x = x_{\text{pred}} + K \cdot y, \qquad P = (1 - K) \cdot P_{\text{pred}}$$

---

<small><b>Intuicja wzmocnienia</b> </small>($K$):<br>

* <small>Gdy sensor mocno szumi </small>$(R \gg P_{\text{pred}}) \implies K \to 0$<small>: ignorujemy pomiar, polegamy na odometrii </small>$(x \approx x_{\text{pred}})$<small>.</small>
* <small>Gdy sensor jest bardzo precyzyjny </small>$(R \ll P_{\text{pred}}) \implies K \to 1$<small>: ufamy pomiarowi </small>$(x \approx z)$<small>.</small>
* <small>Każda poprawna korekta zmniejsza niepewność stanu: </small>$(1 - K) \le 1 \implies P < P_{\text{pred}}$<small>.</small>


**Zadanie 4.1 – Krok predykcji i korekty Filtru Kalmana (1D)**

<small>Uzupełnij implementację kroku predykcji oraz korekty Filtru Kalmana dla pozycji robota </small>$x$<small> na podstawie wzorów powyżej:</small>


In [ ]:
def kalman_predict(x, P, dx, Q):
    """
    Krok predykcji (predict) Filtru Kalmana 1D dla pozycji robota.
    x: aktualna szacowana pozycja robota
    P: wariancja niepewności pozycji
    dx: przemieszczenie robota z odometrii
    Q: wariancja szumu procesu (ruchu)
    """
    x_new = ...
    P_new = ...
    return x_new, P_new


def kalman_update(x, P, z, R):
    """
    Krok korekty (update) Filtru Kalmana 1D dla pozycji robota.
    x: szacowana pozycja robota (przed korektą, z kroku predykcji)
    P: wariancja niepewności pozycji (przed korektą)
    z: pomiar pozycji z czujnika
    R: wariancja szumu pomiarowego
    """
    y = ...
    K = ...
    x_new = ...
    P_new = ...
    return x_new, P_new


**Zadanie 4.2 – Filtr Kalmana w ruchu wahadłowym (1D)**

<small>Wykorzystaj zaimplementowane w Zadaniu 4.1 funkcje `kalman_predict` i `kalman_update` w pętli sterowania. Użyj ruchu wahadłowego z Zadania 3: w każdym kroku pętli wykonaj predykcję z odometrii oraz korektę stanu czujnikiem podczerwieni (IR).</small>

In [ ]:
robot.reset_pose()
clock = Clock(duration=Time.s(10))

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
d0_ir = robot.ir_front.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
d0_lidar = robot.lidar.beam_front.meters
rec = SensorRecorder(
    clock,
    title="Filtr Kalmana 1D (IR) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"ir": "scatter", "kalman": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    ir=lambda: d0_ir - robot.ir_front.distance.meters,
    kalman=lambda: x,
    lidar=lambda: d0_lidar - robot.lidar.beam_front.meters,
)

with LivePlot(rec), robot:
    for t in clock:
        # Wklej tutaj ruch wahadłowy z Zadania 3:
        ...

        # Predykcja:
        curr_odom_x = robot.pose.x.meters
        dx = ...
        last_odom_x = curr_odom_x
        x, P = kalman_predict(x, P, dx, Q)

        # Pomiar:
        z = ...

        # Korekta:
        x, P = kalman_update(x, P, z, R_ir)


### Fuzja wielu czujników – w uproszczeniu

<small>Napisany przez nas algorytm Filtru Kalmana pozwala uwzględnić tylko jedno źródło. Z reguły tych źródeł będziemy posiadać wiele – w końcu nasz robot posiada aż sześć czujników oraz skaner laserowy.</small>

<small>Natomiast funkcja w takiej postaci bardzo dobrze nadaje się do nauki, a dołożenie kolejnego czujnika jest tak proste, jak wykonanie sekwencyjnie kolejnych aktualizacji Filtrem Kalmana:</small>

```python
x, P = kalman_predict(x, P, dx, Q)     # Krok predykcji (ruch z odometrii)
x, P = kalman_update(x, P, z_ir, R_ir) # Filtr Kalmana z wejściem IR
x, P = kalman_update(x, P, z_us, R_us) # Filtr Kalmana z wejściem IR oraz US
```

$$\text{Predykcja}(Q) \;\longrightarrow\; \text{Korekta}(z_{\text{IR}}, R_{\text{IR}}) \;\longrightarrow\; \text{Korekta}(z_{\text{US}}, R_{\text{US}})$$


**Zadanie 4.3 – Sekwencyjna fuzja czujników (IR + US)**

<small>Skopiuj kod z Zadania 4.2 i dodaj sekwencyjne wywołanie `kalman_update` dla sensora US nałożone na IR.</small>

<small>Na wspólnym wykresie zestawiamy: odometrię, stan po IR (`kalman_ir`), fuzję obu czujników (`kalman_fusion`) oraz LiDAR jako niezależny punkt odniesienia.</small>

In [ ]:
robot.reset_pose()
clock = Clock(duration=Time.s(10))

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
d0_ir = robot.ir_front.distance.meters
d0_us = robot.us_front.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
x_ir = 0.0
x_fusion = 0.0
d0_lidar = robot.lidar.beam_front.meters
rec = SensorRecorder(
    clock,
    title="Sekwencyjna fuzja (IR + US) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"kalman_ir": "line", "kalman_fusion": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    kalman_ir=lambda: x_ir,
    kalman_fusion=lambda: x_fusion,
    lidar=lambda: d0_lidar - robot.lidar.beam_front.meters,
)

with LivePlot(rec), robot:
    for t in clock:
        ...  # rozpocznij od skopiowania zawartości zadania 4.2


<div class="alert alert-block alert-info">
  <small><b>Pytanie do przemyślenia:</b> Co gdyby czujnik był zamontowany <b>pod kątem</b>, a nie wprost w kierunku ściany? Czy wówczas również moglibyśmy użyć odczytu bezpośrednio (<i>z</i> &minus; <i>x</i>), czy filtr potrzebuje wiedzieć, w jakiej relacji do osi ruchu znajduje się dany sensor? Odpowiedź na to pytanie znajdziesz poniżej.</small>
</div>


### Pełnoprawna fuzja sensorów (Kalman 2D)

<small>Do tej pory stosowaliśmy uproszczenie w 1D. Pełnoprawny Filtr Kalmana radzi sobie z fuzją wielu źródeł znacznie wydajniej – operuje wprost na rachunku macierzowym, korygując cały stan robota w jednym kroku obliczeniowym.</small>

<small>Bezpośrednim wejściem do algorytmu są macierze, których wymiary wynikają z rozmiaru wektora stanu (</small>$n = 2$<small>) oraz liczby sygnałów pomiarowych (</small>$m = 6$<small>):</small>

* **Stan robota** $\mathbf{x} = \begin{bmatrix} x \\ y \end{bmatrix}$ <small>&ndash; współrzędne pozycji 2D robota na arenie (wymiar </small>$n = 2$<small>).</small>
* **Szum procesu** $\mathbf{Q}_{2 \times 2} = \operatorname{diag}([Q, Q])$ <small>&ndash; niepewność ruchu z odometrii na pojedynczy krok (znamy z Zadania 2).</small>
* **Szum pomiaru** $\mathbf{R}_{6 \times 6} = \operatorname{diag}([R, \dots, R])$ <small>&ndash; szumy pomiarowe 6 sensorów (znamy z Zadania 3).</small>
* **Macierz obserwacji** $\mathbf{H}_{6 \times 2}$ <small>&ndash; określa, <b>w jaki sposób każdy sensor wpływa na stan</b> (</small>$\mathbf{z} = \mathbf{H}\mathbf{x} + \mathbf{v}$<small>). Każdy wiersz reprezentuje jedno źródło pomiaru, a kolumny przypisują sensor do mierzonej współrzędnej (</small>$x$<small> lub </small>$y$<small>):</small>

$$\mathbf{H} = \left. \begin{bmatrix} 
1 & 0 \\
\vdots & \vdots 
\end{bmatrix} \right\} \text{6 wierszy (czujników)}
\quad
\begin{matrix} 
\leftarrow \text{czujnik przedni (wzrost } x\text{)} \\
\dots 
\end{matrix}$$

<small>Skaner LiDAR nie bierze udziału w filtracji – posłuży na wykresie jako niezależny punkt odniesienia.</small>


**Zadanie 5 – Fuzja odometrii i 6 czujników pokładowych**

<small>Dokończ definicję wektora źródeł pomiarowych </small>`z`<small> (odczyty znormalizowane o odległość bazową </small>$d_0$<small>) oraz uzupełnij macierze </small>`Q_mat`<small>, </small>`R_mat`<small> i </small>`H`<small>:</small>


In [ ]:
d0_ir_f, d0_ir_l, d0_ir_r = None, None, None
d0_us_f, d0_us_l, d0_us_r = None, None, None

def sensors_reset():
    global d0_ir_f, d0_ir_l, d0_ir_r, d0_us_f, d0_us_l, d0_us_r
    d0_ir_f, d0_ir_l, d0_ir_r = robot.ir_front.distance.meters, robot.ir_left.distance.meters, robot.ir_right.distance.meters
    d0_us_f, d0_us_l, d0_us_r = robot.us_front.distance.meters, robot.us_left.distance.meters, robot.us_right.distance.meters

robot.reset_pose()
sensors_reset()

z = [
    lambda: ...,  # IR przód
    lambda: ...,  # IR lewy
    lambda: ...,  # IR prawy
    lambda: ...,  # US przód
    lambda: ...,  # US lewy
    lambda: ...,  # US prawy
]

Q_mat = np.diag([..., ...])

R_mat = np.diag([..., ..., ..., ..., ..., ...])

H = np.array([
    ...,  # IR przód
    ...,  # IR lewy
    ...,  # IR prawy
    ...,  # US przód
    ...,  # US lewy
    ...,  # US prawy
])


<small>Uruchamiamy przejazd testowy na trasie kwadratu. Rejestrujemy jednocześnie na jednym wykresie dryfującą odometrię, stan robota z Filtru Kalmana 2D (fuzja 6 czujników) oraz niezależny punkt odniesienia z LiDAR-u:</small>


In [ ]:
robot.reset_pose()
lidar_reset()
sensors_reset()
kalman = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)
rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    kalman=lambda: kalman.state,
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria, Filtr Kalmana, LiDAR"
)
with LivePlot(rec), robot:
    for _ in range(3):
        drive_square()


### <small>Co się dzieje przy złym strojeniu parametrów?</small>

<small>Wzmocnienie Kalmana </small>$K$<small> zależy od stosunku niepewności ruchu </small>$Q$<small> do szumu pomiaru </small>$R$<small>:</small>
* <small><b>Za duże </b></small>$R$<small><b> (lub za małe </b></small>$Q$<small><b>):</b> filtr nie ufa pomiarom, polega na odometrii i przejmuje jej narastający dryf.</small>
* <small><b>Za małe </b></small>$R$<small><b> (lub za duże </b></small>$Q$<small><b>):</b> filtr ufa bezkrytycznie każdemu odczytowi &rarr; stan robota drży od szumu aparatury.</small>

<small>Uruchamiamy trzy filtry 2D jednocześnie podczas jednego przejazdu po kwadracie:</small>
* <small>`kf_optimal` &ndash; dobrze dobrane parametry (stabilny, domknięty kwadrat),</small>
* <small>`kf_trust_odom` &ndash; </small>$R \times 1000$<small> (filtr ignoruje czujniki i dryfuje z kołami),</small>
* <small>`kf_trust_sensor` &ndash; </small>$R \times 0{,}001$<small> (filtr jest nerwowy i szarpie pozycją przy każdym szumie czujników).</small>


In [ ]:
robot.reset_pose()
sensors_reset()

kf_optimal      = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)
kf_trust_odom   = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat * 1000, H=H)
kf_trust_sensor = Kalman2d(robot, z=z, Q=Q_mat * 1000, R=R_mat * 0.001, H=H)

rec_tuning_2d = PoseRecorder(
    optimal=lambda: kf_optimal.state,
    trust_odom=lambda: kf_trust_odom.state,
    trust_sensor=lambda: kf_trust_sensor.state,
    title="Wpływ strojenia Q i R w 2D"
)

with LivePlot(rec_tuning_2d), robot:
    for _ in range(3):
        drive_square()
